In [ ]:
import numpy as np
import pickle 
import matplotlib.pylab as plt
from matplotlib.ticker import FuncFormatter
from scipy.interpolate import interp1d
import h5py
from numpy.fft import rfft, rfftfreq
import astropy
import astropy.units as u
import astropy.coordinates as ac
from scipy.optimize import curve_fit
import copy
import gridder_edit as gridder
from single_freq_edit import Solver as SSolver
from multiprocessing import Queue
from threading import Thread
from pylab import cm
import gc

In [ ]:
def psd(x):
    return np.abs(rfft(x)) ** 2

def use_mm_naive(intv, bits, todallwo, todallho, todallfg, todallfgfn, point, solutionwo, \
                     solutionho, solutionfg, solutionfgfn, queue):
#def use_mm_naive(intv,bits, todallwo, todallho, todallw, todallf, point, solutionwo, \
#                     solutionho, solutionw, solutionf):
    """ use the map maker """
    
    star = bits[intv]
    enl = bits[intv+1]
    
    for xno in range(star, enl):
        
        print(star, enl, xno)
        
        thetodwo = todallwo[:,:,xno]
        thetodho = todallho[:,:,xno]
        thetodfg = todallfg[:,:,xno]
        thetodfgfn = todallfgfn[:,:,xno]

        #initialise using first dish
        sw=SSolver.naive(thetodwo[0,:], point[0])
        for dd in range(1,np.shape(thetodwo)[0]):
            sw.with_naive_obs(thetodwo[dd,:], point[0]) 
        solutionwo[:, xno], _ = sw.solve()
        
        #initialise using first dish
        sh=SSolver.naive(thetodho[0,:], point[0])
        for dd in range(1,np.shape(thetodho)[0]):
            sh.with_naive_obs(thetodho[dd,:], point[0]) 
        solutionho[:, xno], _ = sh.solve()
        
        #initialise using first dish
        sfg=SSolver.naive(thetodfg[0,:], point[0])
        for dd in range(1,np.shape(thetodfg)[0]):
            sfg.with_naive_obs(thetodfg[dd,:], point[0])    
        solutionfg[:, xno], _ = sfg.solve()
        
        #initialise using first dish
        sfgfn=SSolver.naive(thetodfgfn[0,:], point[0])
        for dd in range(1,np.shape(thetodfgfn)[0]):
            sfgfn.with_naive_obs(thetodfgfn[dd,:], point[0])    
        solutionfgfn[:, xno], _ = sfgfn.solve()

    #return solutionwo[:, star:enl], solutionho[:, star:enl], solutionw[:, star:enl], \
    #        solutionf[:, star:enl], star, enl
    queue.put([solutionwo[:, star:enl], solutionho[:, star:enl], \
                   solutionfg[:, star:enl], solutionfgfn[:, star:enl], star, enl])
    
def fn_form(thex, rnlevel, fknee, alpha):
    return rnlevel * (1 + (fknee/thex)**alpha)

def fit_fn_to_data(xdata, ydata, initial_params):
    
    x0 = initial_params
    lower_bounds = [1.e-12, 1.e-12, 1.0]
    upper_bounds = [1.e12, 1.e12, 2.0]
    param_bounds = [lower_bounds, upper_bounds]

    fitted_params, pcov = curve_fit(fn_form, xdata, ydata, x0, \
                                   bounds=param_bounds, maxfev=1e6)
    rmsnoise, thefn, tha = fitted_params
   
    return rmsnoise, thefn, tha 
    
def use_mm(intv, bits, todall, point, itnum, solution2, queue):
#def use_mm(intv, bits, todall, point, itnum, solution2):
    """ use the map maker """
    
    star = bits[intv]
    enl = bits[intv+1]
    
    for xno in range(star, enl):
        
        print(star, enl, xno)
        
        thetod = todall[:,:,xno]

        #initialise using first dish
        s1=SSolver.naive(thetod[0,:], point[0])
        #first make naive map
        for dd in range(1,np.shape(thetod)[0]):
            s1.with_naive_obs(thetod[dd,:], point[0])    
        solution1, res1 = s1.solve()

        ndish = int(np.shape(thetod)[0])
        blockn = 16
        nscan = int(np.shape(thetod)[1] / blockn)
        
        for ii in range(itnum):
        
            thecovs = np.zeros_like(thetod)
        
            for dd in range(ndish):
                for ss in range(0, np.shape(thetod)[1] - 100, nscan):
                    nse = copy.deepcopy(np.array(res1)[dd, ss:ss+nscan])

                    freqs = np.fft.rfftfreq(len(nse), d=2.0)
                    theps = psd(nse)
                    guesspl = np.percentile(theps, 50)
                    nval, fkval, aval = fit_fn_to_data(freqs[1:], theps[1:], \
                                                np.array([guesspl, 0.0002, 1.39]))
                    smops = fn_form(freqs, nval, fkval, aval)
                    #can't fit for high 1/f!!!!!
                    #smops = fn_form(freqs, guesspl, 1.0, 1.0)
                    smops[0] = smops[1] + 1.e-3
                    
                    cov = list(np.fft.irfft(smops))
                    if len(nse)!=len(cov):
                        cov=cov[:len(cov)//2]+[0]+cov[len(cov)//2:]
                    thecovs[dd,ss:ss+nscan] = cov

            s2=SSolver(thetod[0,:], point[0], thecovs[0,:])
            s2.block_num = blockn
            s2.iter_max=250 #taking ages so coming down from 1000 to 500
            for dd in range(1,np.shape(thetod)[0]):
                s2.with_obs(thetod[dd,:], point[0], thecovs[dd,:])    
            solution2[:, xno], res1 = s2.solve()
        
    #return solution2[:, star:enl], star, enl
    queue.put([solution2[:, star:enl], star, enl])

In [ ]:
def read_data(loc, loc2, timestring, blurb, blurb2, mockn, ch0, ch1):
    
    hf1 = h5py.File(loc+'sim_wigglez_tod_h1only/'+mockn+blurb+timestring+'.h5', 'r')
    vis1 = np.array(hf1["vis"])
    h1only = 0.5 * (vis1[:,ch0:ch1,0, :] + vis1[:,ch0:ch1,1, :])
    hf1.close()
    del hf1
    gc.collect()
    print('done')
    hf2 = h5py.File(loc+'sim_wigglez_tod_noh1_wn/'+mockn+blurb2+timestring+'.h5', 'r')
    vis2 = np.array(hf2["vis"])
    wnonly = 0.5 * (vis2[:,ch0:ch1,0, :] + vis2[:,ch0:ch1,1, :])
    hf2.close()
    del hf2
    gc.collect()
    print('done')
    #needs to be loc2
    hf3 = h5py.File(loc2+'sim_wigglez_tod_h1_wn_fg/'+mockn+blurb+timestring+'.h5', 'r')
    ra = np.array(hf3["ra"][:,0])
    dec = np.array(hf3["dec"][:,0])
    vis3 = np.array(hf3["vis"])
    h1wnfg = 0.5 * (vis3[:,ch0:ch1,0, :] + vis3[:,ch0:ch1,1, :])
    hf3.close()
    del hf3
    gc.collect()
    print('done')
    #needs to be loc2
    hf6 = h5py.File(loc2+'sim_wigglez_tod_h1_wn_fg_fn/'+mockn+blurb+timestring+'.h5', 'r')
    vis6 = np.array(hf6["vis"])
    h1fnoise = 0.5 * (vis6[:,ch0:ch1,0, :] + vis6[:,ch0:ch1,1, :])
    hf6.close()
    del hf6
    gc.collect()
    print('done')
    
    print('read in 1 block')
    
    return ra, dec, wnonly, h1only, h1wnfg, h1fnoise

In [ ]:
def select_data(blurb, blurb2, mockn, loc, loc2, f0, f1):

    ra1, dec1, wnon1, h1on1, h1fgs1, h1fnoise1 = read_data(loc, loc2, '20190401172431', blurb, blurb2, mockn, f0, f1)
    ra2, dec2, wnon2, h1on2, h1fgs2, h1fnoise2 = read_data(loc, loc2, '20190402222725', blurb, blurb2, mockn, f0, f1)
    ra3, dec3, wnon3, h1on3, h1fgs3, h1fnoise3 = read_data(loc, loc2, '20190403222429', blurb, blurb2, mockn, f0, f1)
    ra4, dec4, wnon4, h1on4, h1fgs4, h1fnoise4 = read_data(loc, loc2, '20190404171523', blurb, blurb2, mockn, f0, f1)
    ra5, dec5, wnon5, h1on5, h1fgs5, h1fnoise5 = read_data(loc, loc2, '20190405221817', blurb, blurb2, mockn, f0, f1)
    ra6, dec6, wnon6, h1on6, h1fgs6, h1fnoise6 = read_data(loc, loc2, '20190407170615', blurb, blurb2, mockn, f0, f1)
    ra7, dec7, wnon7, h1on7, h1fgs7, h1fnoise7 = read_data(loc, loc2, '20190407221215', blurb, blurb2, mockn, f0, f1)
    ra8, dec8, wnon8, h1on8, h1fgs8, h1fnoise8 = read_data(loc, loc2, '20190403171829', blurb, blurb2, mockn, f0, f1)
    ra9, dec9, wnon9, h1on9, h1fgs9, h1fnoise9 = read_data(loc, loc2, '20190404222123', blurb, blurb2, mockn, f0, f1)
    ra10, dec10, wnon10, h1on10, h1fgs10, h1fnoise10 = read_data(loc, loc2, '20190408170319', blurb, blurb2, mockn, f0, f1)
    ra11, dec11, wnon11, h1on11, h1fgs11, h1fnoise11 = read_data(loc, loc2, '20190405171227', blurb, blurb2, mockn, f0, f1)
    ra12, dec12, wnon12, h1on12, h1fgs12, h1fnoise12 = read_data(loc, loc2, '20190401223031', blurb, blurb2, mockn, f0, f1)
    ra13, dec13, wnon13, h1on13, h1fgs13, h1fnoise13 = read_data(loc, loc2, '20190406170921', blurb, blurb2, mockn, f0, f1)
    ra14, dec14, wnon14, h1on14, h1fgs14, h1fnoise14 = read_data(loc, loc2, '20190408220919', blurb, blurb2, mockn, f0, f1)
    ra15, dec15, wnon15, h1on15, h1fgs15, h1fnoise15 = read_data(loc, loc2, '20190406221521', blurb, blurb2, mockn, f0, f1)
    ra16, dec16, wnon16, h1on16, h1fgs16, h1fnoise16 = read_data(loc, loc2, '20190402172125', blurb, blurb2, mockn, f0, f1)
    
    #getting pointing matrix
    dspace = 0.458 #using 0.458 same as 128 healpy code

    #put obs together 
    #ra = np.hstack((ra1, ra2, ra3, ra4, ra5, ra6, ra7, ra8))#, ra9, ra10, ra11, ra12, \
    #                    ra13, ra14, ra15, ra16))
    ra = np.hstack((ra9, ra10, ra11, ra12, ra13, ra14, ra15, ra16))
    #dec = np.hstack((dec1, dec2, dec3, dec4, dec5, dec6, dec7, dec8))#, dec9, dec10, \
    #                     dec11, dec12, dec13, dec14, dec15, dec16))
    dec = np.hstack((dec9, dec10, dec11, dec12, dec13, dec14, dec15, dec16))
    #justwn = np.vstack((wnon1, wnon2, wnon3, wnon4, wnon5, wnon6, wnon7, wnon8))#, \
    #                        wnon9, wnon10, wnon11, wnon12, wnon13, wnon14, \
    #                            wnon15, wnon16))
    justwn = np.vstack((wnon9, wnon10, wnon11, wnon12, wnon13, wnon14, wnon15, wnon16))
    #justh1 = np.vstack((h1on1, h1on2, h1on3, h1on4, h1on5, h1on6, h1on7, h1on8))#, \
    #                        h1on9, h1on10, h1on11, h1on12, h1on13, h1on14, \
    #                            h1on15, h1on16))
    justh1 = np.vstack((h1on9, h1on10, h1on11, h1on12, h1on13, h1on14, h1on15, h1on16))
    #theh1fgs = np.vstack((h1fgs1, h1fgs2, h1fgs3, h1fgs4, h1fgs5, \
    #                             h1fgs6, h1fgs7, h1fgs8))#, h1fgs9, \
                                 #    h1fgs10, h1fgs11, h1fgs12, h1fgs13, \
                               #          h1fgs14, h1fgs15, h1fgs16))
    theh1fgs = np.vstack((h1fgs9, h1fgs10, h1fgs11, h1fgs12, h1fgs13, h1fgs4, h1fgs15, h1fgs16))
    #theh1fnoise = np.vstack((h1fnoise1, h1fnoise2, h1fnoise3, h1fnoise4, h1fnoise5, \
    #                             h1fnoise6, h1fnoise7, h1fnoise8))#, h1fnoise9, \
    #                                 h1fnoise10, h1fnoise11, h1fnoise12, h1fnoise13, \
    #                                     h1fnoise14, h1fnoise15, h1fnoise16))
    theh1fnoise = np.vstack((h1fnoise9, h1fnoise10, h1fnoise11, h1fnoise12, h1fnoise13, \
                                         h1fnoise14, h1fnoise15, h1fnoise16))

    justwn = np.swapaxes(justwn, 0,2)
    justwn = np.swapaxes(justwn, 1,2)
    justh1 = np.swapaxes(justh1, 0,2)
    justh1 = np.swapaxes(justh1, 1,2)
    theh1fgs = np.swapaxes(theh1fgs, 0,2)
    theh1fgs = np.swapaxes(theh1fgs, 1,2)
    theh1fnoise = np.swapaxes(theh1fnoise, 0,2)
    theh1fnoise = np.swapaxes(theh1fnoise, 1,2)
    
    #Define pixels
    skypoints=[]
    skypoints.append(ac.SkyCoord(ra=ra*u.deg, dec=dec*u.deg))
    (p, m) = gridder.define_pixel(skypoints, dspace*u.deg)

    #subtract off receiver temp per frequency 
    ndish = np.shape(justwn)[0]
    justwn = justwn - np.mean(justwn, axis=1)[:,None,:]  
    theh1fgs = theh1fgs - np.mean(theh1fgs, axis=1)[:,None,:]
    theh1fnoise = theh1fnoise - np.mean(theh1fnoise, axis=1)[:,None,:] 
    
    gpix = np.where(ra > 10.)[0]
    print('Number of Pixels:')
    print(len(gpix))

    justwn = justwn[:, gpix, :]
    justh1 = justh1[:, gpix, :]
    theh1fgs = theh1fgs[:, gpix, :]
    theh1fnoise = theh1fnoise[:, gpix, :]
    ra = ra[gpix]
    dec = dec[gpix]
    numpix = len(gpix)
    
    sgoodra1 = np.where(ra1 > 150.)[0]
    sgoodra2 = np.where(ra1 < 175.)[0] 
    sgoodra = np.intersect1d(sgoodra1, sgoodra2)
    sgooddec1 = np.where(dec1 > -2.)[0]
    sgooddec2 = np.where(dec1 < 8.5)[0] 
    sgooddec = np.intersect1d(sgooddec1, sgooddec2)
    sgpix = np.intersect1d(sgoodra, sgooddec)
    
    pdat = p[0].toarray()
    mshape = np.shape(m)[0]
    hits = np.zeros((mshape))
    for mm in range(mshape):
        hits[mm] = np.sum(pdat[:,mm])
    
    xspace = np.linspace(np.amin(m[:,0]), np.amax(m[:,0]), 5)
    yspace = np.linspace(np.amin(m[:,1]), np.amax(m[:,1]), 5)
    xspace1 = np.linspace(np.amin(ra1), np.amax(ra1), 5)
    yspace1 = np.linspace(np.amin(dec1), np.amax(dec1), 5)
    ystart = dec[-1]
    xstart = ra[0]
    ybit = np.linspace(np.amin(dec), np.amax(dec), 5).astype('int')
    xbit = np.linspace(np.amin(ra), np.amax(ra), 5).astype('int')
    
    import matplotlib.patches as patches
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))
    plt.axes(ax1)
    plt.plot(ra1, dec1, 'b.')
    ax1.set_xlim(ax1.get_xlim()[::-1])
    ax1.tick_params(axis='both', which='major', labelsize=13)
    ax1.set_yticks(yspace1)
    ax1.set_yticklabels(ybit)
    ax1.set_xticks(xspace1)
    ax1.set_xticklabels(xbit)
    plt.xlabel(r'R.A ($^{\circ}$)', fontsize=14)
    plt.ylabel(r'Dec ($^{\circ}$)', fontsize=14)
    plt.axes(ax2)
    plt.scatter(m[:,0], m[:,1], s=80, c=hits, zorder=1)
    ax2.set_xlim(ax2.get_xlim()[::-1])
    plt.colorbar().set_label(label='No. of Hits',size=14)
    ax2.set_yticks(yspace)
    ax2.set_yticklabels(ybit)
    ax2.set_xticks(xspace)
    ax2.set_xticklabels(xbit)
    ax2.tick_params(axis='both', which='major', labelsize=13)
    plt.xlabel(r'R.A ($^{\circ}$)', fontsize=14)
    plt.ylabel(r'Dec ($^{\circ}$)', fontsize=14)
    rarange = np.amax(ra) - np.amin(ra)
    mrange = np.amax(m[:,0]) - np.amin(m[:,0])
    rat = rarange / mrange
    firra = np.amin(m[:,0]) + (np.amax(ra) - 175)/rat
    spacra = 25.0/rat
    ax2.add_patch(patches.Rectangle((firra, np.amin(m[:,1])), spacra, np.amax(m[:,1]) - np.amin(m[:,1]), \
                                    linewidth=3, edgecolor='plum', facecolor='none', zorder=2))
    #plt.savefig('sims_region.pdf', format='pdf', bbox_inches='tight')
    plt.show()
    
    chop = 5#5 for 1MHz
    
    new_w = np.zeros((ndish, numpix, int(np.shape(justwn)[2]/chop)))
    new_h1 = np.zeros((ndish, numpix, int(np.shape(justwn)[2]/chop)))
    new_fg = np.zeros((ndish, numpix, int(np.shape(justwn)[2]/chop)))
    new_f = np.zeros((ndish, numpix, int(np.shape(justwn)[2]/chop)))
    
    
    for dd in range(ndish):
        fno = 0
        for ff in range(0, np.shape(justwn)[2], chop):
            new_w[dd, :, fno] = np.mean(justwn[dd, :, ff:ff+chop], axis=1)
            new_h1[dd, :, fno] = np.mean(justh1[dd, :, ff:ff+chop], axis=1)
            new_fg[dd, :, fno] = np.mean(theh1fgs[dd, :, ff:ff+chop], axis=1)
            new_f[dd, :, fno] = np.mean(theh1fnoise[dd, :, ff:ff+chop], axis=1)
            fno += 1
            
    justwn = new_w
    justh1 = new_h1
    theh1fgs = new_fg
    theh1fnoise = new_f
    
    return justwn, justh1, theh1fgs, theh1fnoise, ra, dec, p, m

In [ ]:
def make_naive_maps(justwn, justh1, theh1fgs, theh1fnoise, ra, dec, p, m):
    
    thds = 16

    # Do one frequency at a time instead of whole cube at once!
    mshape = np.shape(m)[0]
    nch = np.shape(justwn)[2]
    solwnonly = np.zeros((mshape, nch))
    solh1only = np.zeros((mshape, nch))
    solfg = np.zeros((mshape, nch))
    solfn = np.zeros((mshape, nch))
    
    print('Making naive maps')
    
    queue = Queue()
    bits = np.linspace(0, nch, thds).astype('int')
    processes = [Thread(target=use_mm_naive, \
                         args=(intv, bits, justwn, justh1, theh1fgs, theh1fnoise, \
                               p, solwnonly, solh1only, solfg, solfn, queue)) for intv in range(thds-1)]

    for prs in processes:
        prs.start()
    for prs in processes:
        result = queue.get()
        solwnonly[:, result[4]:result[5]] = result[0]
        solh1only[:, result[4]:result[5]] = result[1]
        solfg[:, result[4]:result[5]] = result[2]
        solfn[:, result[4]:result[5]] = result[3]
    for prs in processes:
        prs.join()

    del queue, prs, result
    gc.collect()
    
    f = []
    f.append(p[0])
    
    imw = gridder.fill_map(solwnonly[:,0], m, f, th=0)
    nx = np.shape(imw)[0]
    ny = np.shape(imw)[1]
    nz = nch

    imwo = np.zeros((nx, ny, nz))
    for ii in range(nz):
        imsing = gridder.fill_map(solwnonly[:,ii], m, f, th=0)
        o1, o2 = np.where(np.isnan(imsing))
        if len(o1) > 100:
             print(len(o1))
        imsing[o1, o2] = 0 #need to ask about power reduction by just adding in zeros!!!
        imwo[:,:,ii] = imsing
     
    imho = np.zeros((nx, ny, nz))
    for ii in range(nz):
        imsing = gridder.fill_map(solh1only[:,ii], m, f, th=0)
        o1, o2 = np.where(np.isnan(imsing))
        if len(o1) > 100:
             print(len(o1))
        imsing[o1, o2] = 0 #need to ask about power reduction by just adding in zeros!!!
        imho[:,:,ii] = imsing
        
    imfg = np.zeros((nx, ny, nz))
    for ii in range(nz):
        imsing = gridder.fill_map(solfg[:,ii], m, f, th=0)
        o1, o2 = np.where(np.isnan(imsing))
        if len(o1) > 100:
             print(len(o1))
        imsing[o1, o2] = 0 #need to ask about power reduction by just adding in zeros!!!
        imfg[:,:,ii] = imsing
    
    imf = np.zeros((nx, ny, nz))
    for ii in range(nz):
        imsing = gridder.fill_map(solfn[:,ii], m, f, th=0)
        o1, o2 = np.where(np.isnan(imsing))
        if len(o1) > 100: 
            print(len(o1))
        imsing[o1, o2] = 0
        imf[:,:,ii] = imsing
        
    return imwo, imho, imfg, imf

In [ ]:
def make_mm_maps(theh1fnoise, ra, dec, p, m):
    
    thds = 16

    # Do one frequency at a time instead of whole cube at once!
    mshape = np.shape(m)[0]
    nch = np.shape(theh1fnoise)[2]
    sol_mm_fn = np.zeros((mshape, nch))
    its = 2 #iterations for noise
    
    print('Making weighted maps')
    
    queue = Queue()
    bits = np.linspace(0, nch, thds).astype('int')
    processes = [Thread(target=use_mm, \
                         args=(intv, bits, theh1fnoise, p, its, sol_mm_fn, \
                                   queue)) for intv in range(thds-1)]

    for prs in processes:
        prs.start()
    for prs in processes:
        result = queue.get()
        sol_mm_fn[:, result[1]:result[2]] = result[0]
    for prs in processes:
        prs.join()

    del queue, prs, result
    gc.collect()
    
    f = []
    f.append(p[0])
    
    im_w = gridder.fill_map(sol_mm_fn[:,0], m, f, th=0)
    nx = np.shape(im_w)[0]
    ny = np.shape(im_w)[1]
    nz = nch
    
    im_mm_f = np.zeros((nx, ny, nz))
    for ii in range(nz):
        imsing = gridder.fill_map(sol_mm_fn[:,ii], m, f, th=0)
        o1, o2 = np.where(np.isnan(imsing))
        if len(o1) > 100:
             print(len(o1))
        imsing[o1, o2] = 0 #need to ask about power reduction by just adding in zeros!!!
        im_mm_f[:,:,ii] = imsing
        
    return im_mm_f

In [ ]:
blurb = 'MeerKAT6_HorizonRasterDrift_ideal_withbeam_'
blurb2 = 'MeerKAT6_HorizonRasterDrift_ideal_None_'
loc = '/idia/users/mirfan/sim_meerkat_test/postrev_1of/HRD_5arcmin/rawf_mult_nch1000/'
loc2 = '/idia/users/mirfan/sim_meerkat_test/postrev_1of/HRD_5arcmin/rawf_mult_nch1000/'
nums = ['000', '001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '011', '012', '013', \
           '014', '015', '016', '017', '018', '019']

#files to large we need to save out 500 channels at a time
chans = np.array([0, 200, 400, 600, 800, 1000])

for mm in range(11, 12):#len(nums)):
    
    mockn = 'sim_mock'+nums[mm]+'/'
    print(mockn)
    
    for cc in range(len(chans)-1):

        justwn, justh1, theh1fg, theh1fnoise, ra, dec, p, m = select_data(blurb, blurb2, mockn, loc, \
                                                                                 loc2, chans[cc], chans[cc+1])
        if __name__ == '__main__':
            imwo, imho, imfg, imf = make_naive_maps(justwn, justh1, theh1fg, theh1fnoise, ra, dec, p, m)
    
        if cc == 0:
            plt.imshow(np.mean(imho, axis=2) - np.mean(imho))
            plt.colorbar()
            plt.title('h1')
            plt.show()

            plt.imshow(np.mean(imfg, axis=2))
            plt.colorbar()
            plt.title('h1+noise+fgs')
            plt.show()
        
        if __name__ == '__main__':
            im_mm_f = make_mm_maps(theh1fnoise, ra, dec, p, m)
    
    
        fig, ((ax1, ax2)) = plt.subplots(nrows=1, ncols=2, figsize=(8, 6))
        plt.axes(ax1)
        plt.imshow(im_mm_f[:,:,0])
        plt.colorbar()
        plt.title('MM')
        plt.axes(ax2)
        plt.imshow(imf[:,:,0])
        plt.title('Binning')
        plt.colorbar()
        plt.show()
    
        theout = '/idia/users/mirfan/sim_meerkat_test/postrev_1of/HRD_5arcmin/rawf_mult_nch1000/maps/split/'
    
        np.save(theout+'sim'+nums[mm]+'_h1+wn+fg+fn_MM_hdr5_map_chlot'+str(chans[cc])+'_las.npy', im_mm_f)
        np.save(theout+'sim'+nums[mm]+'_h1+wn+fg+fn_hdr5_map_chlot'+str(chans[cc])+'_las.npy', imf)
        np.save(theout+'sim'+nums[mm]+'_h1+wn+fg_hdr5_map_chlot'+str(chans[cc])+'_las.npy', imfg)
        np.save(theout+'sim'+nums[mm]+'_wn_hdr5_map_chlot'+str(chans[cc])+'_las.npy', imwo)
        np.save(theout+'sim'+nums[mm]+'_h1_hdr5_map_chlot'+str(chans[cc])+'_las.npy', imho)
    
        if  mm == 0 and cc == 0:
            np.save(theout+'ra_las.npy', ra)
            np.save(theout+'dec_las.npy', dec)
            
        del ra, dec, imho, imwo, imfg, imf, im_mm_f
        gc.collect()
        